# Secure AI Services for Research and How to Make your Data Depositable in a FAIR way

**FDZ Mannheim** | Research Skills – Research Data Management Seminars

[Thomas Schmidt](https://orcid.org/0000-0003-3620-3355) & [Jan Kamlah](https://orcid.org/0000-0002-0417-7562)

2026-10-27

## **Task**: Metadata schema mapping with GenAI

In the following example we will use the raw text description of a synthetic data set to create structured metadata using generative AI (i.e., a Large Language Model).

We will:

1. load the description found in `../data/dataset_description_synthetic.txt`
2. load a metadata schema (Dublin Core Terms) found in `../templates/dc_terms.xml`
3. inject both files into a `PROMPT` and
4. task a LLM running on the ChatAI API Service to perform the metadata mapping

We start with an introduction of the API Service.

---

## **ChatAI** API Service: Prerequisites

<img src="./assets/chatai_logo.png" alt="ChatAI logo" width="15%">

Before you start using the service, carefully read the [ChatAI FAQs](https://docs.hpc.gwdg.de/services/ai-services/chat-ai/faq/index.html), the [Data Privacy Notice](https://docs.hpc.gwdg.de/services/ai-services/chat-ai/data-privacy/index.html) and the [Terms of Use](https://docs.hpc.gwdg.de/services/ai-services/chat-ai/terms_of_use.de/index.html).

### Requesting an API key

In order to use the ChatAI API Service you need: 
1. An [Academic Cloud account](https://docs.gwdg.de/doku.php?%20id=de:services:general_services:academicid:start)
2. An API key for ChatAI

You can requests a key at [https://kisski.gwdg.de/en/leistungen/2-02-llm-service/](https://kisski.gwdg.de/en/leistungen/2-02-llm-service/) by clicking on the **Book** icon and filling out the contact form. You should receive your API key after a couple of days. 

### Managing your API keys

Delete and create new ones via the [SAIA dashboard](https://saia.gwdg.de/dashboard).

### ChatAI docs

Find code examples for working with the ChatAI API Service on CLI and in `Python` on this site: [https://docs.hpc.gwdg.de/services/ai-services/saia/index.html#api-request](https://docs.hpc.gwdg.de/services/ai-services/saia/index.html#api-request)

---

## **maKI** API Services: Prerequisites

### Requesting an API key

[https://maki.uni-mannheim.de/docs/api-key-antrag/](https://maki.uni-mannheim.de/docs/api-key-antrag/)

### maKI docs

- Docs: [https://maki.uni-mannheim.de/docs/](https://maki.uni-mannheim.de/docs/)
- Quick Start: [https://maki.uni-mannheim.de/docs/guides/quickstart/](https://maki.uni-mannheim.de/docs/guides/quickstart/)

## Using the API Service

Both ChatAI and maKI use OpenAI compatible API endpoints. We can therfore query both services using the offical `openai` [pip package](https://pypi.org/project/openai/) and install it first.

In [ ]:
%pip install openai

We start by importing the necessary libraries and loading our API key from our environment.

In [ ]:
import os
from pathlib import Path

from openai import OpenAI

In [ ]:
# ChatAI: uncomment the next line and set CHATAI_API_KEY in your environment
# api_key = os.getenv("CHATAI_API_KEY")

api_key = os.getenv("MAKI_API_KEY")  

## **<span style="color:green">🤝 Responsible Use of Generative AI</span>**

<img src="./assets/coderefinery_risks.png" alt="Sloppy API key users" width="80%">

🔗 [https://coderefinery.github.io/coding-with-ai/security/](https://coderefinery.github.io/coding-with-ai/security/)

### API key safety

<img src="./assets/api_chaos.png" alt="Sloppy API key users" width="40%">

#### Do
- Think of API keys as passwords for your sensitive data

#### Don't
- Never *hardcode* API keys in jupyter notebooks or scripts
- Never *commit* them to public Git repositories (for example Github)
- Never *share* API keys via chat, email, screenshots, or public forums
- Make sure AI agents cannot *access* `.env`, `/secrets` etc. where you store your projects' API keys

#### Common pitfalls
- *Notebook outputs*: a printed key (or an error traceback containing it) is saved in the `.ipynb` file. Clear outputs before committing.
- *Git history*: deleting a key in a later commit doesn't remove it from history. Revoke the key and create a new one.
- *Public Git repos*: bots scan GitHub for leaked keys within minutes

### Checklist from CodeRefinery

Source: [https://coderefinery.github.io/coding-with-ai/security/#short-checklist](https://coderefinery.github.io/coding-with-ai/security/#short-checklist)

Before relying on AI-assisted code:

- [ ] I can explain what the relevant code does.
- [ ] It recovers known answers on synthetic or reference data, and I checked N at each transformation.
- [ ] Dependencies are legitimate packages I verified.
- [ ] I can reproduce the environment and workflow (versions, parameters, seeds).
- [ ] Substantive AI use is disclosed where required.

Before giving an AI tool access:

- [ ] I know what information it can see and where that information is processed.
- [ ] That information is appropriate for this tool, account, and service.
- [ ] Credentials and sensitive data are outside its reach.
- [ ] It has only the filesystem, network, and execution privileges it needs.
- [ ] Potentially destructive work is version-controlled, backed up, or sandboxed.

---

We now create a client to query the ChatAI API endpoint and test if our API key works by listing all available ChatAI models.

In [ ]:
client = OpenAI(
    api_key = api_key,
    # base_url = "https://chat-ai.academiccloud.de/v1"  # Base URL for ChatAI API Service
    base_url = "https://maki.uni-mannheim.de/v1"  # Base URL for maKI API Service
)

In [ ]:
sorted([m.id for m in client.models.list()])

The maki docs provide HuggingFace 🤗 links to each [available model](https://maki.uni-mannheim.de/docs/guides/models/). Refer to the model cards for more information on model capabilities.

<img src="./assets/maki_model_list.png" alt="maKI model list" width="70%">

We select a model ...

In [ ]:
model = "qwen3.8-27b-thinking"

... and send a simple API request using the `chat.completions` endpoint of our client.

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are an expert in automating research data management tasks."
        },
        {
            "role": "user",
            "content": "Explain the FAIR principles in max. 4 sentences."
        }
    ],
)

In [ ]:
print(response.choices[0].message.content.strip())

We now read the description of our `dataset` and the metadata schema (`DC Terms`).

In [ ]:
# Filepaths
data_desc_fpath = Path("../data/dataset_description_synthetic.txt")
dc_terms_fpath = Path("../templates/dc_terms.xml")

In [ ]:
def load_fpaths(fpath: Path) -> str:
    with open(fpath, "r") as file:
        return file.read()

data_desc = load_fpaths(data_desc_fpath)
dc_terms_schema = load_fpaths(dc_terms_fpath)

In [ ]:
print(data_desc)

In [ ]:
print(dc_terms_schema)

We define a prompt with a task and inject both the dataset description and our metadata schema.

In [ ]:
PROMPT = f"""Map the following <dataset_description> to Dublin Core terms by using the provided <dc_terms_schema>.
Return ONLY a valid XML and nothing else.

<dataset_description>
{data_desc}
</dataset_description>

<dc_terms_schema>
{dc_terms_schema}
</dc_terms_schema>"""

In [ ]:
# print(PROMPT)

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You are an expert in automating research data management tasks."
        },
        {
            "role": "user",
            "content": PROMPT
        }
    ],
)

In [ ]:
print(response.choices[0].message.content.strip())

Lastly, we save the LLM response to disk.

In [ ]:
output_dir = Path("output")
output_fpath = output_dir / "metadata_mapping.xml"

if not output_dir.exists():
    output_dir.mkdir(exist_ok=True, parents=True)

with open(output_fpath, "w", encoding="utf-8") as file:
    file.write(response.choices[0].message.content.strip())

---

## References

CodeRefinery, Glerean, E., Lindi, B., Pöhner, I., Rantaharju, J., Christ, S., & Pfau, T. (2026, August 10). Responsible Use of Generative AI in Assisted Coding - CodeRefinery lesson. Zenodo. [https://doi.org/10.5281/zenodo.21877460](https://doi.org/10.5281/zenodo.21877460)

European Commission, Directorate-General for Research and Innovation. (2026). Living Guidelines on the Responsible Use of Generative AI in Research (3rd Aufl.). [https://research-and-innovation.ec.europa.eu/document/download/2b6cf7e5-36ac-41cb-aab5-0d32050143dc_en?filename=ec_rtd_ai-guidelines.pdf](https://research-and-innovation.ec.europa.eu/document/download/2b6cf7e5-36ac-41cb-aab5-0d32050143dc_en?filename=ec_rtd_ai-guidelines.pdf)

Kaufmann, R., & Sidiropoulos, C. (2026, September 8). Coding with AI: A Spectrum Between Vibe Coding and Software Engineering. Zenodo. [https://doi.org/10.5281/zenodo.22659083](https://doi.org/10.5281/zenodo.22659083)

Kaufmann, R. (2026, September 18). Safe and secure use of AI in research projects. Zenodo. [https://doi.org/10.5281/zenodo.22829420](https://doi.org/10.5281/zenodo.22829420)